# Multi-Factor Authentication (MFA) – Solution

**Extended Project** – pure-Python TOTP + password hashing + lockout + backup codes + security simulation.



## Process Flowchart
![MFA Flowchart](mfa_flowchart.png)


## 1. Imports


In [ ]:
import hmac
import hashlib
import struct
import time
import base64
import secrets
import math
import random
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

print("Modules ready")


## 2. Password hashing (PBKDF2)


In [ ]:
def hash_password(password, salt=None):
    if salt is None:
        salt = secrets.token_bytes(16)
    dk = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    return salt.hex(), dk.hex()

def verify_password(password, salt_hex, hash_hex):
    salt = bytes.fromhex(salt_hex)
    dk = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    return hmac.compare_digest(dk.hex(), hash_hex)

# Demo
s, h = hash_password("S3cureP@ss!")
print("Hash OK:", verify_password("S3cureP@ss!", s, h))
print("Wrong  :", verify_password("wrong", s, h))


## 3. TOTP secret


In [ ]:
def generate_totp_secret(nbytes=20):
    raw = secrets.token_bytes(nbytes)
    return base64.b32encode(raw).decode('ascii').rstrip('=')

print("Sample secret:", generate_totp_secret())


## 4. HOTP / TOTP / verify (RFC 6238)


In [ ]:
def _b32decode(secret):
    secret = secret.upper().replace(' ', '')
    pad = '=' * ((8 - len(secret) % 8) % 8)
    return base64.b32decode(secret + pad)

def hotp(secret, counter, digits=6):
    key = _b32decode(secret)
    msg = struct.pack('>Q', counter)
    digest = hmac.new(key, msg, hashlib.sha1).digest()
    offset = digest[-1] & 0x0F
    code_int = struct.unpack('>I', digest[offset:offset+4])[0] & 0x7FFFFFFF
    return str(code_int % (10 ** digits)).zfill(digits)

def totp(secret, digits=6, interval=30, for_time=None):
    if for_time is None:
        for_time = time.time()
    counter = int(for_time // interval)
    return hotp(secret, counter, digits)

def verify_totp(secret, code, window=1, digits=6, interval=30, for_time=None):
    if for_time is None:
        for_time = time.time()
    code = str(code).strip().zfill(digits)
    for delta in range(-window, window + 1):
        expected = totp(secret, digits, interval, for_time + delta * interval)
        if hmac.compare_digest(expected, code):
            return True
    return False

# Demo
sec = generate_totp_secret()
now_code = totp(sec)
print("Current TOTP:", now_code)
print("Verify OK   :", verify_totp(sec, now_code))
print("Verify bad  :", verify_totp(sec, "000000"))


## 5. Backup codes


In [ ]:
def generate_backup_codes(n=8, length=8):
    alphabet = 'ABCDEFGHJKLMNPQRSTUVWXYZ23456789'
    return [''.join(secrets.choice(alphabet) for _ in range(length)) for _ in range(n)]

print(generate_backup_codes(4))


## 6. UserStore (full MFA)


In [ ]:
class UserStore:
    def __init__(self):
        self.users = {}

    def register(self, username, password, enable_mfa=True):
        if username in self.users:
            raise ValueError("Username already exists")
        salt, pw_hash = hash_password(password)
        secret = generate_totp_secret() if enable_mfa else None
        backups = generate_backup_codes() if enable_mfa else []
        self.users[username] = {
            'salt': salt, 'hash': pw_hash,
            'totp_secret': secret, 'mfa_enabled': enable_mfa,
            'backup_codes': backups,
            'failed_attempts': 0, 'locked_until': 0.0,
        }
        return {'username': username, 'totp_secret': secret,
                'backup_codes': backups.copy(), 'mfa_enabled': enable_mfa}

    def authenticate(self, username, password, otp=None, use_backup=False,
                     max_attempts=5, lockout_seconds=300):
        user = self.users.get(username)
        if not user:
            return False, "Unknown user"
        now = time.time()
        if user['locked_until'] > now:
            return False, f"Account locked. Try again in {int(user['locked_until']-now)}s"
        if not verify_password(password, user['salt'], user['hash']):
            user['failed_attempts'] += 1
            if user['failed_attempts'] >= max_attempts:
                user['locked_until'] = now + lockout_seconds
                user['failed_attempts'] = 0
                return False, "Too many failures – account locked"
            return False, "Invalid password"
        if not user['mfa_enabled']:
            user['failed_attempts'] = 0
            return True, "Login successful (MFA disabled)"
        if use_backup and otp:
            code = otp.upper()
            if code in user['backup_codes']:
                user['backup_codes'].remove(code)
                user['failed_attempts'] = 0
                return True, "Login successful (backup code used)"
            user['failed_attempts'] += 1
            return False, "Invalid backup code"
        if otp is None or not verify_totp(user['totp_secret'], otp):
            user['failed_attempts'] += 1
            if user['failed_attempts'] >= max_attempts:
                user['locked_until'] = now + lockout_seconds
                user['failed_attempts'] = 0
                return False, "Too many failures – account locked"
            return False, "Invalid OTP"
        user['failed_attempts'] = 0
        return True, "Login successful (MFA verified)"

print("UserStore ready")


## 7. End-to-end demo


In [ ]:
store = UserStore()
info = store.register("alice", "S3cureP@ss!", enable_mfa=True)
print("Registered :", info['username'])
print("TOTP secret:", info['totp_secret'])
print("Backup codes:", info['backup_codes'][:3], "...")

current = totp(info['totp_secret'])
print("\nCurrent OTP:", current)
ok, msg = store.authenticate("alice", "S3cureP@ss!", current)
print("Auth result :", ok, "–", msg)

ok2, msg2 = store.authenticate("alice", "wrongpassword", current)
print("Bad password:", ok2, "–", msg2)

bc = info['backup_codes'][0]
ok3, msg3 = store.authenticate("alice", "S3cureP@ss!", bc, use_backup=True)
print("Backup login:", ok3, "–", msg3)


## Alternate Implementation – dictionary + free functions


In [ ]:
# Pure functional style (no class)
_users = {}

def register_user(username, password, enable_mfa=True):
    if username in _users:
        raise ValueError("exists")
    salt, h = hash_password(password)
    secret = generate_totp_secret() if enable_mfa else None
    _users[username] = dict(salt=salt, hash=h, totp_secret=secret,
                            mfa_enabled=enable_mfa, backup_codes=generate_backup_codes() if enable_mfa else [],
                            failed_attempts=0, locked_until=0.0)
    return secret

def login(username, password, otp=None):
    u = _users.get(username)
    if not u or not verify_password(password, u['salt'], u['hash']):
        return False, "fail"
    if u['mfa_enabled'] and (otp is None or not verify_totp(u['totp_secret'], otp)):
        return False, "bad otp"
    return True, "ok"

sec = register_user("carol", "demo123")
print("Functional login:", login("carol", "demo123", totp(sec)))


## More Practice – Solutions


In [ ]:
# 1. Disable MFA
def disable_mfa(store, username, password):
    u = store.users.get(username)
    if not u or not verify_password(password, u['salt'], u['hash']):
        return False, "auth failed"
    u['mfa_enabled'] = False
    u['totp_secret'] = None
    u['backup_codes'] = []
    return True, "MFA disabled"

# 2. otpauth URI (for QR code / authenticator apps)
def otpauth_uri(secret, account="user@example.com", issuer="MyApp"):
    return f"otpauth://totp/{issuer}:{account}?secret={secret}&issuer={issuer}&algorithm=SHA1&digits=6&period=30"

print("URI example:", otpauth_uri(info['totp_secret'], "alice@demo"))

# 3. Simple rate-limit dict (per-username timestamps)
recent_attempts = defaultdict(list)
def is_rate_limited(username, max_per_minute=10):
    now = time.time()
    recent_attempts[username] = [t for t in recent_attempts[username] if now - t < 60]
    if len(recent_attempts[username]) >= max_per_minute:
        return True
    recent_attempts[username].append(now)
    return False

print("Rate-limit check (first):", is_rate_limited("alice"))


## Simulation Section


In [ ]:
# === SIMULATION PARAMETERS ===
N_TRIALS = 400
WINDOWS = [0, 1, 2, 3]
ATTEMPTS = [1, 5, 10, 30, 100]
# =============================

random.seed(2026)
np.random.seed(2026)

success_rates = {}
for w in WINDOWS:
    for att in ATTEMPTS:
        successes = 0
        for _ in range(N_TRIALS):
            secret = generate_totp_secret()
            guesses = [f"{random.randint(0,999999):06d}" for _ in range(att)]
            t0 = 1_700_000_000.0
            if any(verify_totp(secret, g, window=w, for_time=t0) for g in guesses):
                successes += 1
        success_rates[(w, att)] = successes / N_TRIALS

print("Success rates (window=1):", {a: round(success_rates[(1,a)]*100, 3) for a in ATTEMPTS})


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("MFA / TOTP Security Simulation (6-digit, 30 s)", fontsize=14, fontweight='bold')

ax = axes[0, 0]
for w in WINDOWS:
    rates = [success_rates[(w, a)] * 100 for a in ATTEMPTS]
    ax.plot(ATTEMPTS, rates, 'o-', label=f'±{w}', lw=2)
ax.set_xlabel("Guesses per time step")
ax.set_ylabel("Success probability (%)")
ax.set_title("Brute-force success rate")
ax.legend(fontsize=8)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
emp = [success_rates[(1, a)] * 100 for a in ATTEMPTS]
theo = [100 * (1 - (1 - 3/1e6)**a) for a in ATTEMPTS]
ax.plot(ATTEMPTS, emp, 's-', label='Empirical', color='#1565C0', lw=2)
ax.plot(ATTEMPTS, theo, '--', label='Theory', color='#C62828', lw=2)
ax.set_xlabel("Guesses per time step")
ax.set_ylabel("Success %")
ax.set_title("Window=±1 : Empirical vs Theory")
ax.legend(fontsize=8)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ws = list(range(0, 6))
ax.bar(ws, [2*w+1 for w in ws], color='#7B1FA2', alpha=0.85, edgecolor='black')
ax.set_xlabel("Window (± steps)")
ax.set_ylabel("# valid codes")
ax.set_title("Accepted code space")
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1, 1]
bits = [math.log2(1e6 / (2*w + 1)) for w in ws]
ax.plot(ws, bits, 'D-', color='#2E7D32', lw=2, markersize=7)
ax.axhline(20, color='orange', ls='--', label='~20-bit')
ax.set_xlabel("Window (± steps)")
ax.set_ylabel("Effective bits")
ax.set_title("Effective security of 6-digit TOTP")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mfa_simulation.png", dpi=140, bbox_inches='tight', facecolor='white')
plt.show()
print("Chart saved → mfa_simulation.png")


## Key Observations
- A 6-digit TOTP with the default ±1 window still leaves ~18–19 bits of effective security – far stronger than a short password but not invulnerable to a high-volume online attack.
- Expanding the acceptance window increases usability (clock skew) at the cost of a linear reduction in security.
- Account lockout after a handful of failures is essential; without it an attacker can keep guessing forever.
- Backup codes must be single-use and stored hashed in a real system.


## Summary
You now have a complete, standards-based MFA implementation that can be dropped into any educational project (including the earlier Random Password Generator).  
Experiment with the simulation parameters to see the security / usability trade-offs for yourself.
